In [ ]:
import scanpy as sc
import pandas as pd
import os

In [ ]:
files = {
    "pig_kidney": "adata_snRNA_pig_processed.h5ad",
    "human_all": "adata_snRNA_human_all_processed.h5ad",
    "human_TNK": "adata_snRNA_human_TNK_processed.h5ad"
}


In [ ]:
# 检查靶基因列表（不区分大小写）
target_genes_pig = ['S100A6', 'COLEC11', 'SPP1', 'CXCL9', 'CXCL10', 'CCL2', 'PECAM1', 'VWF', 'CDH5']
target_genes_human = ['CXCR3', 'CCR2', 'CD68', 'CD163', 'ITGAM', 'CXCL9', 'CXCR3']

In [ ]:
for key, filename in files.items():
    if not os.path.exists(filename):
        print(f"❌ 未找到文件: {filename}，请检查路径。")
        continue
        
    print("="*60)
    print(f"📊 正在读取数据: {filename} ({key})")
    print("="*60)
    
    # 1. 读取数据
    adata = sc.read_h5ad(filename)
    print(f"• 细胞数 (Cells): {adata.n_obs}")
    print(f"• 基因数 (Genes): {adata.n_vars}")
    
    # 2. 查看元数据列名 (临床时间点 POD、细胞类型等信息通常存放在这里)
    print("\n• 元数据列名 (obs columns):")
    print(list(adata.obs.columns))
    
    # 3. 尝试寻找“细胞类型”和“时间点”列
    # 常见命名包括 'cell_type', 'celltype', 'cell_type_fine', 'ident', 'timepoint', 'sample', 'group' 等
    possible_cell_cols = ['cell_type', 'celltype', 'cell_type_fine', 'cell_type_broad', 'ident', 'Annotation']
    possible_time_cols = ['timepoint', 'time', 'sample', 'group', 'POD', 'stage']
    
    cell_col = next((c for c in possible_cell_cols if c in adata.obs.columns), None)
    time_col = next((t for t in possible_time_cols if t in adata.obs.columns), None)
    
    if cell_col:
        print(f"\n• 细胞亚群分布 (基于 '{cell_col}'):")
        print(adata.obs[cell_col].value_counts().head(10))  # 打印前10个最多的细胞类型
    else:
        print("\n⚠️ 未找到明显的细胞类型标注列，请手动检查 adata.obs.head()")
        print(adata.obs.head(3))
        
    if time_col:
        print(f"\n• 时间点分布 (基于 '{time_col}'):")
        print(adata.obs[time_col].value_counts())
        
    # 4. 检查我们的关键基因是否存在于 var_names 中
    print("\n• 关键靶基因检测:")
    test_genes = target_genes_pig if "pig" in key else target_genes_human
    existing_genes = []
    missing_genes = []
    # 考虑猪的基因名可能大写或有前缀，做不区分大小写匹配
    var_names_upper = [g.upper() for g in adata.var_names]
    
    for tg in test_genes:
        if tg.upper() in var_names_upper:
            # 找到原始基因名
            idx = var_names_upper.index(tg.upper())
            existing_genes.append(adata.var_names[idx])
        else:
            missing_genes.append(tg)
            
    print(f"  - 已找到的靶基因: {existing_genes}")
    if missing_genes:
        print(f"  - ⚠️ 未找到的靶基因: {missing_genes} (可能是命名不同或未表达)")
        
    # 5. 检查降维图是否存在 (UMAP)
    if 'X_umap' in adata.obsm.keys():
        print("\n• 已包含 UMAP 坐标，可直接绘图。")
    else:
        print("\n⚠️ 未检测到 UMAP 坐标 ('X_umap' missing in obsm)")
    print("\n")

In [ ]:
import scanpy as sc
import pandas as pd

# 1. 读取人源免疫细胞全集数据
print("正在读取人源细胞数据...")
adata_human = sc.read_h5ad("adata_snRNA_human_all_processed.h5ad")
print(f"数据读取成功！共包含 {adata_human.n_obs} 个细胞。")

# 2. 查看 cell_type1 的完整统计（包括 others）
print("\n" + "="*50)
print("📊 cell_type1 完整细胞类型分布:")
print("="*50)
print(adata_human.obs['cell_type1'].value_counts())

# 3. 查看 cell_type2 的完整统计（可能包含更细分的单核/巨噬亚群）
if 'cell_type2' in adata_human.obs.columns:
    print("\n" + "="*50)
    print("📊 cell_type2 完整细胞类型分布:")
    print("="*50)
    print(adata_human.obs['cell_type2'].value_counts())

# 4. 交叉分析：看看非 'others' 的细胞都在哪些时间点（time_point）出现
print("\n" + "="*50)
print("⏰ 不同时间点非 others 细胞的分布情况:")
print("="*50)
# 排除 others 后的交叉表
non_others = adata_human[adata_human.obs['cell_type1'] != 'others']
if len(non_others) > 0:
    ct_time_table = pd.crosstab(non_others.obs['cell_type1'], non_others.obs['time_point'])
    print(ct_time_table)
else:
    print("⚠️ 警告：所有细胞的 cell_type1 均为 'others'？")
    # 如果全是others，我们打印前30行的具体内容来排查
    print(adata_human.obs[['sample_id', 'time_point', 'cell_type1', 'cell_type2']].head(30))

# 5. 顺便打印一下前30个基因名，帮我们彻底搞清楚基因命名的格式
print("\n" + "="*50)
print("🧬 基因名称格式样例（前30个基因）:")
print("="*50)
print(list(adata_human.var_names[:30]))


In [ ]:
import scanpy as sc
import pandas as pd

# 1. 读取人源细胞数据
print("正在读取人源细胞数据...")
adata_human = sc.read_h5ad("adata_snRNA_human_all_processed.h5ad")
print("读取完成！\n")

# 2. 生成 cell_type2 与 time_point 的交叉分布表
# 列出每个时间点下，各类细胞的具体数量
ct2_time_dist = pd.crosstab(
    index=adata_human.obs['cell_type2'], 
    columns=adata_human.obs['time_point'],
    margins=True, # 增加 Total 总计列
    margins_name="Total"
)

print("="*80)
print("📊 cell_type2 在不同时间点 (time_point) 的细胞数量分布表:")
print("="*80)
# 设置 pandas 显示选项以防列数太多被折叠
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(ct2_time_dist)
print("="*80)

# 3. 额外计算占比（百分比），看看每种细胞在不同时间点的构成比
ct2_time_pct = pd.crosstab(
    index=adata_human.obs['cell_type2'], 
    columns=adata_human.obs['time_point'],
    normalize='index' # 按行归一化，即每种细胞在各时间点的分布比例
) * 100

print("\n📊 各细胞类型在不同时间点的分布比例 (%) :")
print(round(ct2_time_pct, 2))


In [ ]:
import scanpy as sc
import os
import pandas as pd
from scipy.sparse import csr_matrix

def export_to_seurat(adata, output_dir):
    """将 AnnData 导出为 Seurat 易读的 mtx 格式"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    print(f"正在导出至 {output_dir}...")
    
    # 1. 导出基因表达矩阵 (转换为稀疏矩阵以节省空间)
    # 确保表达矩阵是稀疏格式
    matrix = adata.X
    if not isinstance(matrix, csr_matrix):
        from scipy.sparse import coo_matrix
        matrix = coo_matrix(matrix).tocsr()
        
    import scipy.io
    scipy.io.mmwrite(os.path.join(output_dir, "matrix.mtx"), matrix.T) # Seurat需要基因作为行，细胞作为列，所以要转置 .T
    
    # 2. 导出条形码 (Barcodes/Cells)
    pd.DataFrame(adata.obs_names).to_csv(os.path.join(output_dir, "barcodes.tsv"), index=False, header=False)
    
    # 3. 导出基因名 (Features/Genes)
    # 很多单细胞数据有基因ID和基因Symbol，我们这里存为两列
    features = pd.DataFrame(index=adata.var_names)
    if 'gene_ids' in adata.var.columns:
        features['gene_ids'] = adata.var['gene_ids']
    else:
        features['gene_ids'] = adata.var_names # 如果没有，就重复基因名
    features['gene_symbol'] = adata.var_names
    features.to_csv(os.path.join(output_dir, "features.tsv"), sep="\t", index=False, header=False)
    
    # 4. 导出元数据 (Metadata)
    adata.obs.to_csv(os.path.join(output_dir, "metadata.csv"))
    print(f"🎉 {output_dir} 导出成功！\n")

# ---- 执行导出 ----
# 1. 导入人源细胞数据
print("读取人源细胞数据...")
adata_human = sc.read_h5ad("adata_snRNA_human_all_processed.h5ad")
export_to_seurat(adata_human, "./seurat_import_human")

# 2. 导入猪源细胞数据 (如果文件在当前目录下)
if os.path.exists("adata_snRNA_pig_processed.h5ad"):
    print("读取猪源细胞数据...")
    adata_pig = sc.read_h5ad("adata_snRNA_pig_processed.h5ad")
    export_to_seurat(adata_pig, "./seurat_import_pig")
else:
    print("未在当前目录下找到 adata_snRNA_pig_processed.h5ad，仅导出了人源数据。")
